# LlamaIndex Demo

RAG pipeline using LlamaIndex v0.10+ with Gemini as the LLM and a HuggingFace BGE model for embeddings.

In [ ]:
!pip install -q \n    llama-index \n    llama-index-llms-gemini \n    llama-index-embeddings-huggingface \n    pypdf docx2txt sentence-transformers

## Imports

In [ ]:
import os
import getpass
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    Settings,
    load_index_from_storage,
)
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from IPython.display import Markdown, display

## Configure LLM & Embeddings

 is the global configuration object that replaced the deprecated  in v0.10.

In [ ]:
# Set your Google API key — getpass avoids hard-coding credentials
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")

Settings.llm          = Gemini(model="models/gemini-1.5-flash")
Settings.embed_model  = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.chunk_size   = 800
Settings.chunk_overlap = 20

## Load Data

Drop PDF, DOCX, TXT, or any supported files into the  folder.

In [ ]:
!mkdir -p data

documents = SimpleDirectoryReader("data").load_data()
print(f"Loaded {len(documents)} document(s)")

In [ ]:
documents

## Build Vector Index

In [ ]:
index = VectorStoreIndex.from_documents(documents)

## Persist & Reload Index

Save the index to disk so you can reload it without re-embedding.

In [ ]:
# Persist to ./storage
index.storage_context.persist()

# To reload later (instead of re-building):
# storage_context = StorageContext.from_defaults(persist_dir="./storage")
# index = load_index_from_storage(storage_context)

## Q&A

In [ ]:
query_engine = index.as_query_engine()

response = query_engine.query("What is LlamaIndex?")
display(Markdown(f"**{response}"))

## Ask Your Own Question

In [ ]:
question = "Summarise the key points from the documents."

response = query_engine.query(question)
display(Markdown(f"**{response}"))